In [2]:
import pandas as pd
import numpy as np
from understatapi import UnderstatClient

In [3]:
TEAM_NAME = "Tottenham"
SEASONS = ["2022", "2023", "2024", "2025"]

In [4]:
interval_stats = []

with UnderstatClient() as understat:
    for season in SEASONS:

        try:
            context_data = understat.team(team=TEAM_NAME).get_context_data(season=season)
            
            timing_dict = context_data.get("timing", {})

            # 順番を制御する＆扱いやすくするため
            interval_mapping = {
                "1-15": 1,
                "16-30": 2,
                "31-45": 3,
                "46-60": 4,
                "61-75": 5,
                "76+": 6
            }
            
            
            for time_interval, stats in timing_dict.items():
                row= {
                    "season": season,
                    "interval": time_interval,
                    "interval_order": interval_mapping.get(time_interval, 99),
                    "shots": stats.get("shots", 0),
                    "goals": stats.get("goals", 0),
                    "xG": stats.get("xG", 0.0),
                    "shots_against": stats.get("against", {}).get("shots", 0),
                    "goals_against": stats.get("against", {}).get("goals", 0),
                    "xGA": stats.get("against", {}).get("xG", 0.0)
                    
                }
    
                interval_stats.append(row)

        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")

df_raw = pd.DataFrame(interval_stats)

In [5]:
df = df_raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   season          24 non-null     str    
 1   interval        24 non-null     str    
 2   interval_order  24 non-null     int64  
 3   shots           24 non-null     int64  
 4   goals           24 non-null     int64  
 5   xG              24 non-null     float64
 6   shots_against   24 non-null     int64  
 7   goals_against   24 non-null     int64  
 8   xGA             24 non-null     float64
dtypes: float64(2), int64(5), str(2)
memory usage: 1.8 KB


In [15]:
# 各シーズンの総失点数を算出
df["season_total_conceded"] = df.groupby("season")["goals_against"].transform("sum")
# 失点割合（％）を算出
df["conceded_percentage"] = (df["goals_against"] / df["season_total_conceded"]) * 100

# 均等に失点した場合の基準値（100％ ÷ 6グループ）
base_percentage = 100 / 6
# 基準値からのズレを計算
df["conceded_deviation"] = df["conceded_percentage"] - base_percentage

# 実際の失点からxGAを引き算した「ミス・集中力キレの指標」
df["xGA_diff"] = df["goals_against"] - df["xGA"]
# 得点数からxGを引いた値　「決定力の指標」
df["xG_diff"] = df["goals"] - df["xG"]
df.head()

,season,interval,interval_order,shots,goals,xG,shots_against,goals_against,xGA,season_total_conceded,conceded_percentage,conceded_deviation,xGA_diff,xG_diff
0,2022,1-15,1,69,8,7.638104,81,14,10.373608,63,22.222222,5.555556,3.626392,0.361896
1,2022,16-30,2,73,7,7.365678,82,8,5.699483,63,12.698413,-3.968254,2.300517,-0.365678
2,2022,31-45,3,66,9,7.997466,90,9,9.709004,63,14.285714,-2.380952,-0.709004,1.002534
3,2022,46-60,4,117,13,14.285550,103,11,10.835323,63,17.460317,0.793651,0.164677,-1.285550
4,2022,61-75,5,91,17,11.876871,82,9,7.459653,63,14.285714,-2.380952,1.540347,5.123129
